# 15 — Join Election Results to the K7 Atlas

This notebook joins the OA21-allocated election layer from Notebook 14 to the K7 OA atlas, then aggregates electoral metrics to:

- current WD25 wards
- LAD25 councils
- K7 clusters
- North West ward atlas outputs

It does not create a final target score. It creates the electoral evidence layer needed for later target scoring.


## 15.1 Project paths and inputs


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name.lower() == "notebooks":
    PROJECT_DIR = NOTEBOOK_DIR.parent
else:
    PROJECT_DIR = NOTEBOOK_DIR

ELECTION_DIR = PROJECT_DIR / "data" / "processed" / "election_results"
ALLOC_DIR = ELECTION_DIR / "oa21_allocated_v1"
ATLAS_DIR = PROJECT_DIR / "data" / "processed" / "atlas_outputs_v1"
AGG_DIR = PROJECT_DIR / "data" / "processed" / "aggregations_v1"
OUTPUT_DIR = ELECTION_DIR / "k7_atlas_join_v1"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ALLOCATED_PATH = ALLOC_DIR / "election_results_oa21_allocated_v1.csv"

OA_BASE_CANDIDATES = [
    ATLAS_DIR / "k7_oa_geo_cluster_base_v1.csv",
    AGG_DIR / "k7_oa_geo_cluster_base_v1.csv",
]
WARD_ATLAS_CANDIDATES = [
    ATLAS_DIR / "k7_ward25_atlas_profile_v1.csv",
    ATLAS_DIR / "k7_ward25_named_full_profile_v1_enriched.csv",
    AGG_DIR / "k7_ward25_named_full_profile_v1_enriched.csv",
    AGG_DIR / "k7_ward25_full_profile_v1.csv",
]
CLUSTER_KEY_CANDIDATES = [
    ATLAS_DIR / "k7_cluster_interpretation_key_v1.csv",
    AGG_DIR / "k7_cluster_interpretation_key_v1.csv",
]

OA_BASE_PATH = next((p for p in OA_BASE_CANDIDATES if p.exists()), None)
WARD_ATLAS_PATH = next((p for p in WARD_ATLAS_CANDIDATES if p.exists()), None)
CLUSTER_KEY_PATH = next((p for p in CLUSTER_KEY_CANDIDATES if p.exists()), None)

if not ALLOCATED_PATH.exists():
    raise FileNotFoundError(f"Allocated election file not found: {ALLOCATED_PATH}. Run Notebook 14 first.")
if OA_BASE_PATH is None:
    raise FileNotFoundError("Could not find k7_oa_geo_cluster_base_v1.csv.")
if WARD_ATLAS_PATH is None:
    raise FileNotFoundError("Could not find a ward atlas profile file.")

print("Project directory:", PROJECT_DIR)
print("Allocated election file:", ALLOCATED_PATH)
print("OA base:", OA_BASE_PATH)
print("Ward atlas:", WARD_ATLAS_PATH)
print("Cluster key:", CLUSTER_KEY_PATH)
print("Output directory:", OUTPUT_DIR)


Project directory: c:\Users\keena\Documents\Electoral_Tribes
Allocated election file: c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\oa21_allocated_v1\election_results_oa21_allocated_v1.csv
OA base: c:\Users\keena\Documents\Electoral_Tribes\data\processed\aggregations_v1\k7_oa_geo_cluster_base_v1.csv
Ward atlas: c:\Users\keena\Documents\Electoral_Tribes\data\processed\atlas_outputs_v1\k7_ward25_atlas_profile_v1.csv
Cluster key: c:\Users\keena\Documents\Electoral_Tribes\data\processed\aggregations_v1\k7_cluster_interpretation_key_v1.csv
Output directory: c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\k7_atlas_join_v1


## 15.2 Load data and attach K7 atlas geography to OA rows


In [2]:
def normalise_code(value):
    if pd.isna(value):
        return pd.NA
    value = str(value).strip().upper()
    if value in ["", "NAN", "NONE", "NULL"]:
        return pd.NA
    return value

allocated = pd.read_csv(ALLOCATED_PATH, low_memory=False)
oa_base = pd.read_csv(OA_BASE_PATH, low_memory=False)
ward_atlas = pd.read_csv(WARD_ATLAS_PATH, low_memory=False)

allocated["OA21CD"] = allocated["OA21CD"].map(normalise_code)
oa_base["OA21CD"] = oa_base["OA21CD"].map(normalise_code)

# Keep only the atlas fields needed for aggregation.
oa_keep_cols = [
    "OA21CD", "cluster_id", "population",
    "WD25CD", "WD25NM", "LAD25CD", "LAD25NM",
    "LSOA21CD", "LSOA21NM", "MSOA21CD", "MSOA21NM",
]
oa_keep_cols = [c for c in oa_keep_cols if c in oa_base.columns]

oa_atlas = oa_base[oa_keep_cols].drop_duplicates("OA21CD").copy()

# Avoid confusion between source-area population and atlas OA population.
oa_atlas = oa_atlas.rename(columns={"population": "atlas_oa_population"})

election_oa_atlas = allocated.merge(
    oa_atlas,
    on="OA21CD",
    how="left",
    validate="many_to_one"
)

print("Allocated OA rows:", len(allocated))
print("Joined OA rows:", len(election_oa_atlas))
print("Missing WD25CD after atlas join:", election_oa_atlas["WD25CD"].isna().sum() if "WD25CD" in election_oa_atlas.columns else "WD25CD missing")
print("Missing cluster_id after atlas join:", election_oa_atlas["cluster_id"].isna().sum() if "cluster_id" in election_oa_atlas.columns else "cluster_id missing")

display(election_oa_atlas.head())


Allocated OA rows: 327629
Joined OA rows: 327629
Missing WD25CD after atlas join: 0
Missing cluster_id after atlas join: 0


,result_area_key,source_year,election_year,election_date,council_name,lad_code,ward_name,standard_ward_name,ward_code,source_boundary_year,...,cluster_id,atlas_oa_population,WD25CD,WD25NM,LAD25CD,LAD25NM,LSOA21CD,LSOA21NM,MSOA21CD,MSOA21NM
0,2022|CODE|ADUR|E05007562|BUCKINGHAM,2022,2022,2022-05-05,Adur,E07000223,Buckingham,Buckingham,E05007562,2022,...,2,278,E05007562,Buckingham,E07000223,Adur,E01031339,Adur 002B,E02006535,Adur 002
1,2022|CODE|ADUR|E05007562|BUCKINGHAM,2022,2022,2022-05-05,Adur,E07000223,Buckingham,Buckingham,E05007562,2022,...,4,335,E05007562,Buckingham,E07000223,Adur,E01031339,Adur 002B,E02006535,Adur 002
2,2022|CODE|ADUR|E05007562|BUCKINGHAM,2022,2022,2022-05-05,Adur,E07000223,Buckingham,Buckingham,E05007562,2022,...,1,274,E05007562,Buckingham,E07000223,Adur,E01031338,Adur 002A,E02006535,Adur 002
3,2022|CODE|ADUR|E05007562|BUCKINGHAM,2022,2022,2022-05-05,Adur,E07000223,Buckingham,Buckingham,E05007562,2022,...,1,352,E05007562,Buckingham,E07000223,Adur,E01031338,Adur 002A,E02006535,Adur 002
4,2022|CODE|ADUR|E05007562|BUCKINGHAM,2022,2022,2022-05-05,Adur,E07000223,Buckingham,Buckingham,E05007562,2022,...,2,289,E05007562,Buckingham,E07000223,Adur,E01031340,Adur 002C,E02006535,Adur 002


## 15.3 Cluster names


In [3]:
if CLUSTER_KEY_PATH is not None and CLUSTER_KEY_PATH.exists():
    cluster_key = pd.read_csv(CLUSTER_KEY_PATH)
    if {"cluster_id", "cluster_name"}.issubset(cluster_key.columns):
        cluster_names = dict(zip(cluster_key["cluster_id"], cluster_key["cluster_name"]))
    else:
        cluster_names = {}
else:
    cluster_names = {}

if not cluster_names:
    cluster_names = {
        0: "Student & Transient Youth",
        1: "Rooted Older Homeowners",
        2: "Stable Suburban Professionals",
        3: "Cosmopolitan Young Professional Core",
        4: "Settled Working Families / Skilled Trades Suburbs",
        5: "Settled Diverse Urban Communities",
        6: "Post-Industrial Estates / Deprived Working Communities",
    }

election_oa_atlas["cluster_name"] = election_oa_atlas["cluster_id"].map(cluster_names)

cluster_names


{0: 'Student & Transient Youth',
 1: 'Rooted Older Homeowners',
 2: 'Stable Suburban Professionals',
 3: 'Cosmopolitan Young Professional Core',
 4: 'Settled Working Families / Skilled Trades Suburbs',
 5: 'Settled Diverse Urban Communities',
 6: 'Post-Industrial Estates / Deprived Working Communities'}

## 15.4 Aggregation helpers

This aggregates allocated vote counts upward, then recalculates shares and fragmentation from the aggregated counts.


In [4]:
PARTY_BUCKETS = ["con", "lab", "ld", "green", "reform_ukip_brexit", "independent", "sdp", "other"]

ALLOCATED_NUMERIC_BASE = [
    "allocated_electorate", "allocated_valid_votes", "allocated_ballots", "allocated_invalid_votes",
    "allocated_top_party_votes", "allocated_runner_up_party_votes",
] + [f"allocated_{bucket}_votes" for bucket in PARTY_BUCKETS]


def aggregate_election_layer(df, group_cols, label):
    group_cols = [c for c in group_cols if c in df.columns]
    numeric_cols = [c for c in ALLOCATED_NUMERIC_BASE if c in df.columns]

    if not group_cols:
        raise ValueError(f"No valid group columns for {label}")

    working = df.dropna(subset=group_cols).copy()

    out = (
        working
        .groupby(group_cols, dropna=False, as_index=False)[numeric_cols]
        .sum()
    )

    # Diagnostics / source coverage.
    coverage = (
        working
        .groupby(group_cols, dropna=False, as_index=False)
        .agg(
            contributing_oa_rows=("OA21CD", "count"),
            contributing_result_areas=("result_area_key", "nunique"),
            contributing_source_years=("source_year", lambda s: "; ".join(map(str, sorted(pd.Series(s).dropna().unique())))),
        )
    )

    out = out.merge(coverage, on=group_cols, how="left")

    valid_col = "allocated_valid_votes"

    for bucket in PARTY_BUCKETS:
        votes_col = f"allocated_{bucket}_votes"
        share_col = f"{bucket}_share"
        if votes_col not in out.columns:
            out[votes_col] = 0
        out[share_col] = np.where(out.get(valid_col, 0) > 0, out[votes_col] / out[valid_col], np.nan)

    share_cols = [f"{bucket}_share" for bucket in PARTY_BUCKETS]
    votes_cols = [f"allocated_{bucket}_votes" for bucket in PARTY_BUCKETS]

    # Top and runner-up buckets by allocated vote count.
    def top_two_parties(row):
        votes = row[votes_cols].astype(float)
        votes.index = [c.replace("allocated_", "").replace("_votes", "") for c in votes.index]
        ranked = votes.sort_values(ascending=False)
        return pd.Series({
            "top_party_bucket": ranked.index[0],
            "runner_up_party_bucket": ranked.index[1],
            "top_party_votes_allocated": ranked.iloc[0],
            "runner_up_party_votes_allocated": ranked.iloc[1],
            "margin_votes_allocated": ranked.iloc[0] - ranked.iloc[1],
        })

    top = out.apply(top_two_parties, axis=1)
    out = pd.concat([out, top], axis=1)

    out["margin_pct_allocated"] = np.where(
        out[valid_col] > 0,
        out["margin_votes_allocated"] / out[valid_col],
        np.nan
    )

    shares_matrix = out[share_cols].fillna(0)
    sum_sq = (shares_matrix ** 2).sum(axis=1)

    out["party_fragmentation_index"] = 1 - sum_sq
    out["effective_number_of_parties"] = np.where(sum_sq > 0, 1 / sum_sq, np.nan)
    out["aggregation_label"] = label

    return out


## 15.5 Aggregate to K7 atlas geographies


In [5]:
# Keep source year in the grouped output. This lets you inspect historical layers separately.
ward_year = aggregate_election_layer(
    election_oa_atlas,
    ["WD25CD", "WD25NM", "LAD25CD", "LAD25NM", "source_year"],
    "ward25_by_source_year"
)

lad_year = aggregate_election_layer(
    election_oa_atlas,
    ["LAD25CD", "LAD25NM", "source_year"],
    "lad25_by_source_year"
)

cluster_year = aggregate_election_layer(
    election_oa_atlas,
    ["cluster_id", "cluster_name", "source_year"],
    "k7_cluster_by_source_year"
)

# Also produce all-year combined versions, but treat them as exploratory because they mix different election years.
ward_all_years = aggregate_election_layer(
    election_oa_atlas,
    ["WD25CD", "WD25NM", "LAD25CD", "LAD25NM"],
    "ward25_all_available_years_combined"
)

lad_all_years = aggregate_election_layer(
    election_oa_atlas,
    ["LAD25CD", "LAD25NM"],
    "lad25_all_available_years_combined"
)

cluster_all_years = aggregate_election_layer(
    election_oa_atlas,
    ["cluster_id", "cluster_name"],
    "k7_cluster_all_available_years_combined"
)

print("Ward-year rows:", len(ward_year))
print("LAD-year rows:", len(lad_year))
print("Cluster-year rows:", len(cluster_year))

display(ward_year.head())


Ward-year rows: 12264
LAD-year rows: 620
Cluster-year rows: 28


,WD25CD,WD25NM,LAD25CD,LAD25NM,source_year,allocated_electorate,allocated_valid_votes,allocated_ballots,allocated_invalid_votes,allocated_top_party_votes,...,other_share,top_party_bucket,runner_up_party_bucket,top_party_votes_allocated,runner_up_party_votes_allocated,margin_votes_allocated,margin_pct_allocated,party_fragmentation_index,effective_number_of_parties,aggregation_label
0,E05000932,Ainsdale,E08000014,Sefton,2022,10034.0,4202.0,0.0,0.0,1387.0,...,0.0,con,lab,1387.0,1354.0,33.0,0.007853,0.693558,3.263258,ward25_by_source_year
1,E05000932,Ainsdale,E08000014,Sefton,2023,9973.0,4211.0,0.0,0.0,1497.0,...,0.0,lab,ld,1497.0,1481.0,16.0,0.003800,0.686257,3.187322,ward25_by_source_year
2,E05000932,Ainsdale,E08000014,Sefton,2024,10061.0,3843.0,0.0,0.0,1978.0,...,0.0,ld,lab,1978.0,1193.0,785.0,0.204267,0.618483,2.621117,ward25_by_source_year
3,E05000933,Birkdale,E08000014,Sefton,2022,10091.0,3498.0,0.0,0.0,1518.0,...,0.0,ld,lab,1518.0,1093.0,425.0,0.121498,0.671389,3.043114,ward25_by_source_year
4,E05000933,Birkdale,E08000014,Sefton,2023,9992.0,3535.0,0.0,0.0,1431.0,...,0.0,ld,lab,1431.0,1356.0,75.0,0.021216,0.666254,2.996288,ward25_by_source_year


## 15.6 Latest available election layer per current WD25 ward

This selects the highest `source_year` currently available for each WD25 ward. It may include county electoral-division derived data for some areas. Keep the source-year field visible and review before using it for target scoring.


In [6]:
ward_year_sorted = ward_year.sort_values(["WD25CD", "source_year"], ascending=[True, False])
ward_latest = ward_year_sorted.drop_duplicates("WD25CD", keep="first").copy()
ward_latest["latest_layer_note"] = "latest_available_source_year_after_oa21_crosswalk"

print("Latest ward rows:", len(ward_latest))
display(ward_latest[["WD25CD", "WD25NM", "LAD25NM", "source_year", "top_party_bucket", "margin_pct_allocated"]].head())


Latest ward rows: 7515


,WD25CD,WD25NM,LAD25NM,source_year,top_party_bucket,margin_pct_allocated
2,E05000932,Ainsdale,Sefton,2024,ld,0.204267
5,E05000933,Birkdale,Sefton,2024,lab,0.022615
8,E05000934,Blundellsands,Sefton,2024,lab,0.516864
11,E05000935,Cambridge,Sefton,2024,ld,0.049953
14,E05000936,Church,Sefton,2024,green,0.037367


## 15.7 Join latest election layer to the K7 ward atlas


In [7]:
# Standardise codes before join.
ward_atlas["WD25CD"] = ward_atlas["WD25CD"].map(normalise_code)
ward_latest["WD25CD"] = ward_latest["WD25CD"].map(normalise_code)

# Prefix election columns to avoid colliding with atlas columns.
join_keys = ["WD25CD"]
latest_for_join = ward_latest.copy()
rename_latest = {
    c: f"latest_election_{c}"
    for c in latest_for_join.columns
    if c not in join_keys
}
latest_for_join = latest_for_join.rename(columns=rename_latest)

ward_atlas_election = ward_atlas.merge(
    latest_for_join,
    on="WD25CD",
    how="left",
    validate="one_to_one"
)

print("Ward atlas rows:", len(ward_atlas))
print("Rows with latest election layer:", ward_atlas_election["latest_election_source_year"].notna().sum())
print("Rows missing latest election layer:", ward_atlas_election["latest_election_source_year"].isna().sum())

display(ward_atlas_election.head())


Ward atlas rows: 7572
Rows with latest election layer: 7515
Rows missing latest election layer: 57


,LAD25CD,LAD25NM,WD25CD,WD25NM,population,oa_count,cluster_0_population,cluster_1_population,cluster_2_population,cluster_3_population,...,latest_election_top_party_bucket,latest_election_runner_up_party_bucket,latest_election_top_party_votes_allocated,latest_election_runner_up_party_votes_allocated,latest_election_margin_votes_allocated,latest_election_margin_pct_allocated,latest_election_party_fragmentation_index,latest_election_effective_number_of_parties,latest_election_aggregation_label,latest_election_latest_layer_note
0,E06000001,Hartlepool,E05013038,Burn Valley,7633,26,332,1276,1461,0,...,lab,con,919.0,339.0,580.0,0.344009,0.628035,2.688426,ward25_by_source_year,latest_available_source_year_after_oa21_crosswalk
1,E06000001,Hartlepool,E05013039,De Bruce,8055,25,0,1382,0,0,...,lab,con,758.0,330.0,428.0,0.309696,0.596895,2.480743,ward25_by_source_year,latest_available_source_year_after_oa21_crosswalk
2,E06000001,Hartlepool,E05013040,Fens & Greatham,6381,23,0,4899,0,0,...,independent,lab,883.0,622.0,261.0,0.125662,0.691391,3.240347,ward25_by_source_year,latest_available_source_year_after_oa21_crosswalk
3,E06000001,Hartlepool,E05013041,Foggy Furze,8151,31,0,2809,0,0,...,lab,con,927.0,503.0,424.0,0.265498,0.557289,2.258808,ward25_by_source_year,latest_available_source_year_after_oa21_crosswalk
4,E06000001,Hartlepool,E05013042,Hart,7645,25,0,2118,2796,0,...,lab,con,841.0,629.0,212.0,0.107451,0.684145,3.166007,ward25_by_source_year,latest_available_source_year_after_oa21_crosswalk


## 15.8 North West subset


In [8]:
north_west_lads = [
    "Cheshire East", "Cheshire West and Chester", "Halton", "Warrington",
    "Cumberland", "Westmorland and Furness",
    "Bolton", "Bury", "Manchester", "Oldham", "Rochdale", "Salford",
    "Stockport", "Tameside", "Trafford", "Wigan",
    "Blackburn with Darwen", "Blackpool", "Burnley", "Chorley", "Fylde",
    "Hyndburn", "Lancaster", "Pendle", "Preston", "Ribble Valley",
    "Rossendale", "South Ribble", "West Lancashire", "Wyre",
    "Knowsley", "Liverpool", "Sefton", "St. Helens", "Wirral",
]

lad_col = "LAD25NM" if "LAD25NM" in ward_atlas_election.columns else None

if lad_col:
    nw_ward_atlas_election = ward_atlas_election[ward_atlas_election[lad_col].isin(north_west_lads)].copy()
else:
    nw_ward_atlas_election = pd.DataFrame()

print("North West ward rows:", len(nw_ward_atlas_election))
if len(nw_ward_atlas_election) > 0:
    display(nw_ward_atlas_election[["WD25CD", "WD25NM", "LAD25NM", "latest_election_source_year"]].head())


North West ward rows: 825


,WD25CD,WD25NM,LAD25NM,latest_election_source_year
103,E05013169,Appleton,Halton,2024.0
104,E05013170,Bankfield,Halton,2024.0
105,E05013171,Beechwood & Heath,Halton,2024.0
106,E05013172,Birchfield,Halton,2024.0
107,E05013173,Bridgewater,Halton,2024.0


## 15.9 Save outputs


In [9]:
outputs = {
    "k7_ward25_election_metrics_by_source_year_v1.csv": ward_year,
    "k7_lad25_election_metrics_by_source_year_v1.csv": lad_year,
    "k7_cluster_election_metrics_by_source_year_v1.csv": cluster_year,
    "k7_ward25_election_metrics_all_available_years_v1.csv": ward_all_years,
    "k7_lad25_election_metrics_all_available_years_v1.csv": lad_all_years,
    "k7_cluster_election_metrics_all_available_years_v1.csv": cluster_all_years,
    "k7_ward25_latest_election_metrics_v1.csv": ward_latest,
    "k7_ward25_atlas_with_latest_election_metrics_v1.csv": ward_atlas_election,
}

if len(nw_ward_atlas_election) > 0:
    outputs["k7_north_west_ward25_atlas_with_latest_election_metrics_v1.csv"] = nw_ward_atlas_election

for filename, df in outputs.items():
    path = OUTPUT_DIR / filename
    df.to_csv(path, index=False)
    print("Saved:", path)

# Basic join diagnostic.
join_diag = pd.DataFrame([{
    "ward_atlas_rows": len(ward_atlas),
    "ward_rows_with_latest_election_layer": ward_atlas_election["latest_election_source_year"].notna().sum(),
    "ward_rows_missing_latest_election_layer": ward_atlas_election["latest_election_source_year"].isna().sum(),
    "allocated_oa_rows": len(election_oa_atlas),
    "allocated_result_areas": election_oa_atlas["result_area_key"].nunique(),
}])

join_diag.to_csv(OUTPUT_DIR / "k7_election_atlas_join_diagnostics_v1.csv", index=False)
display(join_diag)


Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\k7_atlas_join_v1\k7_ward25_election_metrics_by_source_year_v1.csv
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\k7_atlas_join_v1\k7_lad25_election_metrics_by_source_year_v1.csv
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\k7_atlas_join_v1\k7_cluster_election_metrics_by_source_year_v1.csv
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\k7_atlas_join_v1\k7_ward25_election_metrics_all_available_years_v1.csv
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\k7_atlas_join_v1\k7_lad25_election_metrics_all_available_years_v1.csv
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\k7_atlas_join_v1\k7_cluster_election_metrics_all_available_years_v1.csv
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\k7_atlas_join_v1\k7_wa

,ward_atlas_rows,ward_rows_with_latest_election_layer,ward_rows_missing_latest_election_layer,allocated_oa_rows,allocated_result_areas
0,7572,7515,57,327629,11592


## 15.10 What this file is / is not

This notebook creates the **electoral evidence layer** for the K7 atlas.

It is suitable for:

- checking which kinds of K7 areas tend to have different party patterns,
- building council and ward-level electoral summaries,
- preparing a later target-scoring model.

It is not yet a final target model. Before target scoring, review:

- 2023 name-matching gaps,
- 2025 county electoral division approximations,
- 2021 historical data exclusion,
- whether the latest available election layer is genuinely comparable across councils.
